In [7]:
import pandas as pd
import csv
import pickle
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer

In [8]:
tokenizer_ger = AutoTokenizer.from_pretrained("dbmdz/bert-base-german-uncased")
tokenizer_eng =  AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

In [9]:
# read in data

path_train = "../Input/Data/train/"

train_task1 = pd.read_json(path_train + "train_task1.json")
eval_task1 = pd.read_json(path_train + "test_task1.json")


path_test = "../Input/Data/test/"

test_flausch = pd.read_csv(path_test + "task1.csv")
test_comments = pd.read_csv(path_test + "comments_extended.csv")
# add flausch labels to comments
test_task1 = pd.merge(test_comments, test_flausch, how="inner")

In [10]:
path = "../Input/Data/"

# original ChatGPT generated lists of positive words, emoticons, emojis
positive_words_orig_csv = pd.read_csv(path + "positive_words.csv")
positive_words_orig = positive_words_orig_csv["Wort"].tolist()

positive_emoticons_csv = pd.read_csv(path + "positive_emoticons.csv")
positive_emoticons = positive_emoticons_csv["Emoticon"].tolist()

positive_emojis_csv = pd.read_csv(path + "positive_emojis.csv")
positive_emojis = positive_emojis_csv["Emoji"].tolist()


path_thesis = "Data/"

# new, longer ChatGPT generated positive word list
positive_words_new_csv = pd.read_csv(path_thesis + "positive_words_new.csv")
positive_words_new = positive_words_new_csv["words"].tolist()

# GermanPolarityClues positive word list
polarity_clues = pd.read_csv(path_thesis + "GermanPolarityClues-Positive-21042012.tsv", sep="\t", header=None)
# get list of only the words
positive_polarity_words = polarity_clues[0].tolist()

# English positive words list 
with open(path_thesis + "positive_words_english.csv", newline='', encoding='utf-8') as infile:
    positive_words_eng = [w for [w] in csv.reader(infile)]

In [11]:
def get01(flausch):
    if flausch == "yes":
        return 1
    else:
        return 0

In [12]:
y_train = []
for i in range(len(train_task1)):

    flausch = train_task1.iloc[i]["flausch"]
    y_train.append(get01(flausch)) 

y_eval = []
for i in range(len(eval_task1)):

    flausch = eval_task1.iloc[i]["flausch"]
    y_eval.append(get01(flausch)) 

y_test = []
for i in range(len(test_task1)):

    flausch = test_task1.iloc[i]["flausch"]
    y_test.append(get01(flausch)) 

# Individual counts

## Word lists

In [13]:
# without comment.lower() for original GPT list

y_pred = []

for i in range(len(eval_task1)):

    comment = str(eval_task1.iloc[i]["comment"])
    flausch_score = 0

    for w in positive_words_orig:
        if w in comment:
            flausch_score += 1

    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

scores = classification_report(y_eval, y_pred, output_dict=True, zero_division=0.0)
print(scores["1"])

{'precision': 0.7492447129909365, 'recall': 0.47509578544061304, 'f1-score': 0.5814771395076201, 'support': 1044.0}


In [14]:
# function for predicting flausch based on word count

def word_count_pred(word_list, comment_version, threshold=0, data=eval_task1, y=y_eval):

    y_pred = []

    for i in range(len(data)):

        comment = str(data.iloc[i][comment_version])
        flausch_score = 0

        for w in word_list:
            if w.lower() in comment.lower():
                flausch_score += 1

        guess = "no"
        if flausch_score > threshold:
            guess = "yes"
        
        y_pred.append(get01(guess))

    scores = classification_report(y, y_pred, output_dict=True, zero_division=0.0)
    print(scores["1"])

In [15]:
# original GPT list, original comment

word_count_pred(positive_words_orig, "comment")

{'precision': 0.7443249701314217, 'recall': 0.5967432950191571, 'f1-score': 0.6624136097820309, 'support': 1044.0}


In [16]:
# original GPT list, spelling corrected comment

word_count_pred(positive_words_orig, "spelling_corrected")

{'precision': 0.740521327014218, 'recall': 0.5986590038314177, 'f1-score': 0.6620762711864409, 'support': 1044.0}


### Shortening word lists

In [17]:
# cut based on frequency not flausch & frequency flausch

def cut_lists(word_list, comment_version, percentage_not):

    not_flausch_comments = train_task1[train_task1["flausch"] == "no"]
    flausch_comments = train_task1[train_task1["flausch"] == "yes"]

    # make dictionary for words in word_list
    not_word_dict = {}
    positive_word_dict = {}
    for w in word_list:
        not_word_dict[w.lower()] = 0
        positive_word_dict[w.lower()] = 0
    
    # count how often words in list occur in not flausch comments
    for i in range(len(not_flausch_comments)):  
        comment = str(not_flausch_comments.iloc[i][comment_version])

        for w in word_list:
            if w.lower() in comment.lower():
                not_word_dict[w.lower()] += 1
    
    # count how often words in list occur in flausch comments
    for i in range(len(flausch_comments)):  
        comment = str(flausch_comments.iloc[i][comment_version])

        for w in word_list:
            if w.lower() in comment.lower():
                positive_word_dict[w.lower()] += 1

    # only include words in less than x percent of not flausch comments
    # but must occur in flausch comments at least once
    word_list_cut = [w for w, freq in not_word_dict.items() 
                    if freq < (len(not_flausch_comments) / (100/percentage_not))
                    and positive_word_dict[w] > 0]
    
    return word_list_cut

In [18]:
word_list = positive_words_orig
comment_version = "comment"

print(len(word_list))
word_count_pred(word_list, comment_version)

# ignore how often words occur in not flausch comments
word_list_cut = cut_lists(word_list, comment_version, 100)
print(len(word_list_cut))
word_count_pred(word_list_cut, comment_version)

# less than 1.5% of not flausch comments
positive_words_orig_cut = cut_lists(word_list, comment_version, 1.5)
print(len(positive_words_orig_cut))
word_count_pred(positive_words_orig_cut, comment_version)

64
{'precision': 0.7443249701314217, 'recall': 0.5967432950191571, 'f1-score': 0.6624136097820309, 'support': 1044.0}
47
{'precision': 0.7443249701314217, 'recall': 0.5967432950191571, 'f1-score': 0.6624136097820309, 'support': 1044.0}
46
{'precision': 0.7801507537688442, 'recall': 0.5948275862068966, 'f1-score': 0.6749999999999999, 'support': 1044.0}


In [20]:
word_list = positive_words_new
comment_version = "comment"

print(len(word_list))
word_count_pred(word_list, comment_version)

word_list_cut = cut_lists(word_list, comment_version, 100)
print(len(word_list_cut))
word_count_pred(word_list_cut, comment_version)

positive_words_new_cut = cut_lists(word_list, comment_version, 1.5)
print(len(positive_words_new_cut))
word_count_pred(positive_words_new_cut, comment_version)

102
{'precision': 0.7093425605536332, 'recall': 0.39272030651340994, 'f1-score': 0.5055487053020962, 'support': 1044.0}
50
{'precision': 0.708838821490468, 'recall': 0.39176245210727967, 'f1-score': 0.5046267735965453, 'support': 1044.0}
49
{'precision': 0.7560521415270018, 'recall': 0.3888888888888889, 'f1-score': 0.5135989879822898, 'support': 1044.0}


In [21]:
word_list = positive_polarity_words
comment_version = "comment"

print(len(word_list))
word_count_pred(word_list, comment_version)

word_list_cut = cut_lists(word_list, comment_version, 100)
print(len(word_list_cut))
word_count_pred(word_list_cut, comment_version)

positive_polarity_words_cut = cut_lists(word_list, comment_version, 1.5)
print(len(positive_polarity_words_cut))
word_count_pred(positive_polarity_words_cut, comment_version)

17627
{'precision': 0.38484848484848483, 'recall': 0.8515325670498084, 'f1-score': 0.530113297555158, 'support': 1044.0}
1225
{'precision': 0.3856832971800434, 'recall': 0.8515325670498084, 'f1-score': 0.5309047476858764, 'support': 1044.0}
1209
{'precision': 0.47735399284862934, 'recall': 0.7672413793103449, 'f1-score': 0.5885378398236591, 'support': 1044.0}


In [22]:
word_list = positive_words_eng
comment_version = "translated"

print(len(word_list))
word_count_pred(word_list, comment_version)

word_list_cut = cut_lists(word_list, comment_version, 100)
print(len(word_list_cut))
word_count_pred(word_list_cut, comment_version)

positive_words_eng_cut = cut_lists(word_list, comment_version, 1.5)
print(len(positive_words_eng_cut))
word_count_pred(positive_words_eng_cut, comment_version)

2006
{'precision': 0.5111260957518543, 'recall': 0.7260536398467433, 'f1-score': 0.5999208547685002, 'support': 1044.0}
414
{'precision': 0.5170765027322405, 'recall': 0.725095785440613, 'f1-score': 0.6036682615629984, 'support': 1044.0}
409
{'precision': 0.564929693961952, 'recall': 0.6542145593869731, 'f1-score': 0.6063027075011096, 'support': 1044.0}


## Token lists

In [23]:
# function for predicting flausch based on token count

def token_count_pred(token_list, tokenizer, comment_version, threshold=0, data=eval_task1, y=y_eval):

    y_pred = []

    for i in range(len(data)):

        comment = str(data.iloc[i][comment_version])
        tokens = tokenizer.tokenize(comment)

        flausch_score = 0

        for t in token_list:
            if t in tokens:
                flausch_score += 1

        guess = "no"
        if flausch_score > threshold:
            guess = "yes"
        
        y_pred.append(get01(guess))

    scores = classification_report(y, y_pred, output_dict=True, zero_division=0.0)
    print(scores["1"])

In [24]:
# function for token list generation

def get_token_list(word_list, tokenizer, comment_version, percentage_not):

    # all tokens of the words in the word list
    token_list = list(set([t for w in word_list for t in tokenizer.tokenize(w)]))

    not_flausch_comments = train_task1[train_task1["flausch"] == "no"]
    flausch_comments = train_task1[train_task1["flausch"] == "yes"]

    # make dictionary for tokens in token_list
    not_token_dict = {}
    positive_token_dict = {}
    for t in token_list:
        not_token_dict[t.lower()] = 0
        positive_token_dict[t.lower()] = 0
    
    # count hot oftem tokens in list occur in not flausch comments
    for i in range(len(not_flausch_comments)):  
        comment = str(not_flausch_comments.iloc[i][comment_version])
        tokens = tokenizer.tokenize(comment)

        for t in token_list:
            if t in tokens:
                not_token_dict[t] += 1
    
    # count how often tokens in list occur in not flausch comments
    for i in range(len(flausch_comments)):  
        comment = str(flausch_comments.iloc[i][comment_version])
        tokens = tokenizer.tokenize(comment)

        for t in token_list:
            if t in tokens:
                positive_token_dict[t] += 1

    # only include tokens in less than x percent of not flausch comments
    # but must occur in flausch comments at least once
    token_list_cut = [t for t, freq in not_token_dict.items() 
                    if freq < (len(not_flausch_comments) / (100/percentage_not))
                    and positive_token_dict[t] > 0]
    
    
    return token_list_cut

In [25]:
# uncut original GPT list

word_list = positive_words_orig
comment_version = "comment"
tokenizer = tokenizer_ger

token_list = get_token_list(word_list, tokenizer, comment_version, 1.25)
print(len(token_list))
token_count_pred(token_list, tokenizer, comment_version)

Token indices sequence length is longer than the specified maximum sequence length for this model (2300 > 512). Running this sequence through the model will result in indexing errors


81
{'precision': 0.6685606060606061, 'recall': 0.6762452107279694, 'f1-score': 0.6723809523809524, 'support': 1044.0}


In [26]:
# cut original GPT list

word_list = positive_words_orig_cut
comment_version = "comment"
tokenizer = tokenizer_ger

token_list = get_token_list(word_list, tokenizer, comment_version, 1.25)
print(len(token_list))
token_count_pred(token_list, tokenizer, comment_version)

57
{'precision': 0.6969376979936642, 'recall': 0.632183908045977, 'f1-score': 0.6629834254143646, 'support': 1044.0}


In [27]:
# uncut new GPT list

word_list = positive_words_new
comment_version = "comment"
tokenizer = tokenizer_ger

token_list = get_token_list(word_list, tokenizer, comment_version, 1.25)
print(len(token_list))
token_count_pred(token_list, tokenizer, comment_version)

88
{'precision': 0.6288951841359773, 'recall': 0.42528735632183906, 'f1-score': 0.5074285714285715, 'support': 1044.0}


In [28]:
# cut new GPT list

word_list = positive_words_new_cut
comment_version = "comment"
tokenizer = tokenizer_ger

token_list = get_token_list(word_list, tokenizer, comment_version, 1.25)
print(len(token_list))
token_count_pred(token_list, tokenizer, comment_version)

59
{'precision': 0.7083333333333334, 'recall': 0.37452107279693486, 'f1-score': 0.4899749373433584, 'support': 1044.0}


In [29]:
# uncut polarity word list

word_list = positive_polarity_words
comment_version = "comment"
tokenizer = tokenizer_ger

token_list = get_token_list(word_list, tokenizer, comment_version, 1.25)
print(len(token_list))
token_count_pred(token_list, tokenizer, comment_version)

1966
{'precision': 0.3687150837988827, 'recall': 0.8850574712643678, 'f1-score': 0.5205633802816901, 'support': 1044.0}


In [30]:
# cut polarity word list

word_list = positive_polarity_words_cut
comment_version = "comment"
tokenizer = tokenizer_ger

token_list = get_token_list(word_list, tokenizer, comment_version, 1.25)
print(len(token_list))
token_count_pred(token_list, tokenizer, comment_version)

877
{'precision': 0.44337606837606836, 'recall': 0.7950191570881227, 'f1-score': 0.5692729766803841, 'support': 1044.0}


In [31]:
# ucnut English list

word_list = positive_words_eng
comment_version = "translated"
tokenizer = tokenizer_eng

token_list = get_token_list(word_list, tokenizer, comment_version, 1.25)
print(len(token_list))
token_count_pred(token_list, tokenizer, comment_version)

Token indices sequence length is longer than the specified maximum sequence length for this model (518 > 512). Running this sequence through the model will result in indexing errors


844
{'precision': 0.41710388247639035, 'recall': 0.7614942528735632, 'f1-score': 0.5389830508474577, 'support': 1044.0}


In [32]:
# cut English list

word_list = positive_words_eng_cut
comment_version = "translated"
tokenizer = tokenizer_eng

token_list = get_token_list(word_list, tokenizer, comment_version, 1.25)
print(len(token_list))
token_count_pred(token_list, tokenizer, comment_version)

395
{'precision': 0.5603305785123966, 'recall': 0.6494252873563219, 'f1-score': 0.6015971606033719, 'support': 1044.0}


### Creating token list from data

In [33]:
def create_token_list(tokenizer, comment_version, percentage_not, percentage_flausch):

    not_flausch_comments = train_task1[train_task1["flausch"] == "no"]
    flausch_comments = train_task1[train_task1["flausch"] == "yes"]


    # count hot often any tokens occur in not flausch comments
    not_token_dict = {}

    for i in range(len(not_flausch_comments)):  
        comment = str(not_flausch_comments.iloc[i][comment_version])
        tokens = tokenizer.tokenize(comment)

        for t in tokens:
            if t in not_token_dict.keys():
                not_token_dict[t] += 1
            else:
                not_token_dict[t] = 1
    
    # count hot often any tokens occur in flausch comments
    positive_token_dict = {}

    for i in range(len(flausch_comments)):  
        comment = str(flausch_comments.iloc[i][comment_version])
        tokens = tokenizer.tokenize(comment)

        for t in tokens:
            if t in positive_token_dict.keys():
                positive_token_dict[t] += 1
            else:
                positive_token_dict[t] = 1

    # any tokens that occur in more than x % of flausch comments 
    # and in less than x% of not flausch comments
    token_list = [t for t, freq in positive_token_dict.items()
                  if freq > (len(flausch_comments) / (100/percentage_flausch))
                  and (t not in not_token_dict.keys()
                  or not_token_dict[t] < (len(not_flausch_comments) / (100/percentage_not)))]
    
    return token_list

In [34]:
comment_version = "comment"
tokenizer = tokenizer_ger

# less than 1.25% of not flausch comments,
# more than 3% of flausch comments
token_list = create_token_list(tokenizer, comment_version, 1.25, 3)
print(len(token_list))
token_count_pred(token_list, tokenizer, comment_version)

23
{'precision': 0.6902071563088512, 'recall': 0.7021072796934866, 'f1-score': 0.6961063627730295, 'support': 1044.0}


## Count functions

In [35]:
# counting how many additional characters are used

def extra_chars(comment):

    extra_count = 0 # count how many additional characters there are
    rep_count = 0 # count how often a character is repeated

    current = ""
    last = ""
    
    for c in comment:
        current = c

        if current == last: # repeated character
            rep_count += 1

            if rep_count > 1: # character repeated more than once
                extra_count += 1
        
        else:
            rep_count = 0 # currently no repetition
            last = current
    
    return extra_count

In [36]:
# original version of all caps
# identifies when at least two consecutive characters are uppercase

def all_caps_old(comment):

    upper_count = 0
    current = ""
    last_upper = False
    in_upper = False

    for c in comment:
        current = c
        
        if current.isupper(): # current character is uppercase

            if last_upper: # previous character was uppercase

                if not in_upper: # second character in a row
                    upper_count += 1
                    in_upper = True
            
        else: # current is not uppercase
            in_upper = False # when uppercase character chain ends
        
        last_upper = current.isupper()
    
    return upper_count

In [37]:
# new version
# identifies entire segments of a comment that are uppercase

def all_caps(comment):

    words = comment.split()

    upper_count = 0

    for w in words:
        if w.isupper():
            # exclude singular uppercase characters
            if len(w) > 1:
                upper_count += 1
    
    return upper_count

In [ ]:
# flausch prediction for old all caps function

y_pred = []

for i in range(len(eval_task1)):

    comment = eval_task1.iloc[i]["comment"]

    flausch_score = all_caps_old(comment)

    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

scores = classification_report(y_eval, y_pred, output_dict=True, zero_division=0.0)
print(scores["1"])

{'precision': 0.33455882352941174, 'recall': 0.0871647509578544, 'f1-score': 0.13829787234042554, 'support': 1044.0}


In [ ]:
# flausch prediction for new all caps function

y_pred = []

for i in range(len(eval_task1)):

    comment = eval_task1.iloc[i]["comment"]

    flausch_score = all_caps(comment)

    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

scores = classification_report(y_eval, y_pred, output_dict=True, zero_division=0.0)
print(scores["1"])

{'precision': 0.3465553235908142, 'recall': 0.15900383141762453, 'f1-score': 0.2179908076165463, 'support': 1044.0}


In [ ]:
# flausch prediction for extra chars function

y_pred = []

for i in range(len(eval_task1)):

    comment = eval_task1.iloc[i]["comment"]

    flausch_score = extra_chars(comment)

    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

scores = classification_report(y_eval, y_pred, output_dict=True, zero_division=0.0)
print(scores["1"])

{'precision': 0.37857142857142856, 'recall': 0.15229885057471265, 'f1-score': 0.21721311475409835, 'support': 1044.0}


## Emoji and emoticon counts

In [41]:
# flausch predictions based on emoji count

y_pred = []

for i in range(len(eval_task1)):

    comment = str(eval_task1.iloc[i]["comment"])
    flausch_score = 0

    for e in positive_emojis:
        if e in comment:
            flausch_score += 1

    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

scores = classification_report(y_eval, y_pred, output_dict=True, zero_division=0.0)
print(scores["1"])

{'precision': 0.3300970873786408, 'recall': 0.13026819923371646, 'f1-score': 0.18681318681318682, 'support': 1044.0}


In [42]:
# flausch predictions based on emoticon count

y_pred = []

for i in range(len(eval_task1)):

    comment = str(eval_task1.iloc[i]["comment"])
    flausch_score = 0

    for e in positive_emoticons:
        if e in comment:
            flausch_score += 1

    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

scores = classification_report(y_eval, y_pred, output_dict=True, zero_division=0.0)
print(scores["1"])

{'precision': 0.4214876033057851, 'recall': 0.19540229885057472, 'f1-score': 0.2670157068062827, 'support': 1044.0}


# Logistic regression

Combining count features with sentiment scores and transformer predictions

In [43]:
# create dataframe with all features for all comments

def get_eval_features(word_list, token_list, tokenizer, comment_version, path=path_train, data=eval_task1):

    # features contain transformer predictions
    features = pd.read_csv(path + "features.csv")
    if path == path_train:
        eval_features = features[features["set"] == "test"].copy()
    else:
        eval_features = features

    sentiment_orig = []
    sentiment_spelling_corrected = []
    sentiment_translated = []
    sentiment_anger = []
    sentiment_disgust = []
    sentiment_fear = []
    sentiment_joy = []
    sentiment_neutral = []
    sentiment_sadness = []
    sentiment_surprise = []
    flausch_words = []
    flausch_tokens = []
    flausch_ratio = []
    flausch_emoticons = []
    flausch_emojis = []
    extra_characters = []
    all_capitalized = []


    for i in range(len(eval_features)):

        id = eval_features.iloc[i]["id"]
    
        orig_comment = data[data["id"] == id]["comment"].item()
        comment = data[data["id"] == id][comment_version].item()
        tokens = tokenizer.tokenize(comment)


        # add sentiments
        sentiment_orig.append(data[data["id"] == id]["sentiment_orig"].item())
        sentiment_spelling_corrected.append(data[data["id"] == id]["sentiment_spelling_corrected"].item())
        sentiment_translated.append(data[data["id"] == id]["sentiment_translated"].item())
        sentiment_anger.append(data[data["id"] == id]["sentiment_anger"].item())
        sentiment_disgust.append(data[data["id"] == id]["sentiment_disgust"].item())
        sentiment_fear.append(data[data["id"] == id]["sentiment_fear"].item())
        sentiment_joy.append(data[data["id"] == id]["sentiment_joy"].item())
        sentiment_neutral.append(data[data["id"] == id]["sentiment_neutral"].item())
        sentiment_sadness.append(data[data["id"] == id]["sentiment_sadness"].item())
        sentiment_surprise.append(data[data["id"] == id]["sentiment_surprise"].item())


        # add counts
        flausch_word_count = 0
        for w in word_list:
            if w.lower() in comment.lower():
                flausch_word_count += 1
        flausch_words.append(flausch_word_count)

        flausch_token_count = 0
        for t in token_list:
            if t in tokens:
                flausch_token_count += 1
        flausch_tokens.append(flausch_token_count)

        ratio = 0
        if len(tokens) != 0:
            ratio = flausch_token_count / len(tokens)
        flausch_ratio.append(ratio)

        emoticon_count = 0
        for e in positive_emoticons:
            if e in orig_comment:
                emoticon_count += 1
        flausch_emoticons.append(emoticon_count)

        emoji_count = 0
        for e in positive_emojis:
            if e in orig_comment:
                emoji_count += 1
        flausch_emojis.append(emoji_count)

        all_capitalized.append(all_caps(orig_comment))
        extra_characters.append(extra_chars(orig_comment))

    
    eval_features["sentiment_orig"] = sentiment_orig
    eval_features["sentiment_spelling_corrected"] = sentiment_spelling_corrected
    eval_features["sentiment_translated"] = sentiment_translated
    eval_features["sentiment_anger"] = sentiment_anger
    eval_features["sentiment_disgust"] = sentiment_disgust
    eval_features["sentiment_fear"] = sentiment_fear
    eval_features["sentiment_joy"] = sentiment_joy
    eval_features["sentiment_neutral"] = sentiment_neutral
    eval_features["sentiment_sadness"] = sentiment_sadness
    eval_features["sentiment_surprise"] = sentiment_surprise
    eval_features["flausch_words"] = flausch_words
    eval_features["flausch_tokens"] = flausch_tokens
    eval_features["flausch_ratio"] = flausch_ratio
    eval_features["flausch_emoticons"] = flausch_emoticons
    eval_features["flausch_emojis"] = flausch_emojis
    eval_features["extra_chars"] = extra_characters
    eval_features["all_caps"] = all_capitalized

    return eval_features

In [44]:
# run logistic regression on evaluation data with train-test split

def log_reg_eval(features, columns, data=eval_task1):

    X = []
    y = []

    for i in range(len(features)):

        id = features.iloc[i]["id"]
        flausch = data[data["id"] == id]["flausch"].item()
        y.append(get01(flausch))

        feature_vec = []

        for c in columns:
            feature_vec.append(features.iloc[i][c])            

        X.append(feature_vec)
    
    train_X, test_X, train_y, test_y =  train_test_split(X, y, test_size=0.2, random_state=42)

    log_reg = LogisticRegression(max_iter=200).fit(train_X, train_y)
    pred_y = log_reg.predict(test_X)
    scores = classification_report(test_y, pred_y, output_dict=True, zero_division=0.0)
    print(scores["1"])

## Comparing lists in combination with all features

In [45]:
# all feature columns

columns = [
        "gbert-large_yes_comment",
        "bert-base-german-cased_yes_comment",
        "bert-base-german-cased_yes_spelling_corrected",
        "gbert-large_yes_spelling_corrected",
        "roberta-large_yes_translated",
        "sentiment_orig",
        "sentiment_spelling_corrected",
        "sentiment_translated",
        "sentiment_anger", 
        "sentiment_disgust",
        "sentiment_fear", 
        "sentiment_joy",
        "sentiment_neutral",
        "sentiment_sadness",
        "sentiment_surprise",
        "flausch_words",
        "flausch_tokens",
        "flausch_ratio",
        "flausch_emoticons",
        "flausch_emojis",
        "extra_chars",
        "all_caps"
        ]

In [46]:
word_list = positive_words_orig
comment_version = "comment"
tokenizer = tokenizer_ger
percentage_not = 1.25

token_list = get_token_list(word_list, tokenizer, comment_version, percentage_not)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)
log_reg_eval(eval_features, columns)

{'precision': 0.9362745098039216, 'recall': 0.9271844660194175, 'f1-score': 0.9317073170731707, 'support': 206.0}


In [47]:
word_list = positive_words_orig_cut
comment_version = "comment"
tokenizer = tokenizer_ger
percentage_not = 1.25

token_list = get_token_list(word_list, tokenizer, comment_version, percentage_not)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)
log_reg_eval(eval_features, columns)

{'precision': 0.9362745098039216, 'recall': 0.9271844660194175, 'f1-score': 0.9317073170731707, 'support': 206.0}


In [48]:
word_list = positive_words_new
comment_version = "comment"
tokenizer = tokenizer_ger
percentage_not = 1.25

token_list = get_token_list(word_list, tokenizer, comment_version, percentage_not)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)
log_reg_eval(eval_features, columns)

{'precision': 0.9310344827586207, 'recall': 0.9174757281553398, 'f1-score': 0.9242053789731051, 'support': 206.0}


In [49]:
word_list = positive_words_new_cut
comment_version = "comment"
tokenizer = tokenizer_ger
percentage_not = 1.25

token_list = get_token_list(word_list, tokenizer, comment_version, percentage_not)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)
log_reg_eval(eval_features, columns)

{'precision': 0.9310344827586207, 'recall': 0.9174757281553398, 'f1-score': 0.9242053789731051, 'support': 206.0}


In [50]:
word_list = positive_polarity_words
comment_version = "comment"
tokenizer = tokenizer_ger
percentage_not = 1.25

token_list = get_token_list(word_list, tokenizer, comment_version, percentage_not)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)
log_reg_eval(eval_features, columns)

{'precision': 0.9356435643564357, 'recall': 0.9174757281553398, 'f1-score': 0.926470588235294, 'support': 206.0}


In [51]:
word_list = positive_polarity_words_cut
comment_version = "comment"
tokenizer = tokenizer_ger
percentage_not = 1.25

token_list = get_token_list(word_list, tokenizer, comment_version, percentage_not)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)
log_reg_eval(eval_features, columns)

{'precision': 0.9356435643564357, 'recall': 0.9174757281553398, 'f1-score': 0.926470588235294, 'support': 206.0}


In [52]:
word_list = positive_words_eng
comment_version = "comment"
tokenizer = tokenizer_ger
percentage_not = 1.25

token_list = get_token_list(word_list, tokenizer, comment_version, percentage_not)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)
log_reg_eval(eval_features, columns)

{'precision': 0.9310344827586207, 'recall': 0.9174757281553398, 'f1-score': 0.9242053789731051, 'support': 206.0}


In [53]:
word_list = positive_words_eng_cut
comment_version = "comment"
tokenizer = tokenizer_ger
percentage_not = 1.25

token_list = get_token_list(word_list, tokenizer, comment_version, percentage_not)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)
log_reg_eval(eval_features, columns)

{'precision': 0.9356435643564357, 'recall': 0.9174757281553398, 'f1-score': 0.926470588235294, 'support': 206.0}


In [ ]:
# best lists from just count predictions
# cut original GPT word list & token list created from data

word_list = positive_words_orig_cut
comment_version = "comment"
tokenizer = tokenizer_ger
percentage_not = 1.25

token_list = create_token_list(tokenizer, comment_version, percentage_not, 3)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)
log_reg_eval(eval_features, columns)

{'precision': 0.9408866995073891, 'recall': 0.9271844660194175, 'f1-score': 0.9339853300733497, 'support': 206.0}


## Combinations of groups of features

In [55]:
word_list = positive_words_orig_cut
comment_version = "comment"
tokenizer = tokenizer_ger

token_list = create_token_list(tokenizer, comment_version, 1.25, 3)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)

In [56]:
columns = [
#        "gbert-large_yes_comment",
#        "bert-base-german-cased_yes_comment",
#        "bert-base-german-cased_yes_spelling_corrected",
#        "gbert-large_yes_spelling_corrected",
#        "roberta-large_yes_translated",
#        "sentiment_orig",
#        "sentiment_spelling_corrected",
#        "sentiment_translated",
#        "sentiment_anger", 
#        "sentiment_disgust",
#        "sentiment_fear", 
#        "sentiment_joy",
#        "sentiment_neutral",
#        "sentiment_sadness",
#        "sentiment_surprise",
        "flausch_words",
        "flausch_tokens",
        "flausch_ratio",
        "flausch_emoticons",
        "flausch_emojis",
        "extra_chars",
        "all_caps"
        ]

log_reg_eval(eval_features, columns)

{'precision': 0.8, 'recall': 0.5825242718446602, 'f1-score': 0.6741573033707865, 'support': 206.0}


In [57]:
columns = [
#        "gbert-large_yes_comment",
#        "bert-base-german-cased_yes_comment",
#        "bert-base-german-cased_yes_spelling_corrected",
#        "gbert-large_yes_spelling_corrected",
#        "roberta-large_yes_translated",
        "sentiment_orig",
        "sentiment_spelling_corrected",
        "sentiment_translated",
        "sentiment_anger", 
        "sentiment_disgust",
        "sentiment_fear", 
        "sentiment_joy",
        "sentiment_neutral",
        "sentiment_sadness",
        "sentiment_surprise",
#        "flausch_words",
#        "flausch_tokens",
#        "flausch_ratio",
#        "flausch_emoticons",
#        "flausch_emojis",
#        "extra_chars",
#        "all_caps"
        ]

log_reg_eval(eval_features, columns)

{'precision': 0.7535211267605634, 'recall': 0.5194174757281553, 'f1-score': 0.6149425287356322, 'support': 206.0}


In [58]:
columns = [
#        "gbert-large_yes_comment",
#        "bert-base-german-cased_yes_comment",
#        "bert-base-german-cased_yes_spelling_corrected",
#        "gbert-large_yes_spelling_corrected",
#        "roberta-large_yes_translated",
        "sentiment_orig",
        "sentiment_spelling_corrected",
        "sentiment_translated",
        "sentiment_anger", 
        "sentiment_disgust",
        "sentiment_fear", 
        "sentiment_joy",
        "sentiment_neutral",
        "sentiment_sadness",
        "sentiment_surprise",
        "flausch_words",
        "flausch_tokens",
        "flausch_ratio",
        "flausch_emoticons",
        "flausch_emojis",
        "extra_chars",
        "all_caps"
        ]

log_reg_eval(eval_features, columns)

{'precision': 0.8363636363636363, 'recall': 0.6699029126213593, 'f1-score': 0.7439353099730458, 'support': 206.0}


In [59]:
columns = [
        "gbert-large_yes_comment",
        "bert-base-german-cased_yes_comment",
        "bert-base-german-cased_yes_spelling_corrected",
        "gbert-large_yes_spelling_corrected",
        "roberta-large_yes_translated",
#        "sentiment_orig",
#        "sentiment_spelling_corrected",
#        "sentiment_translated",
#        "sentiment_anger", 
#        "sentiment_disgust",
#        "sentiment_fear", 
#        "sentiment_joy",
#        "sentiment_neutral",
#        "sentiment_sadness",
#        "sentiment_surprise",
#        "flausch_words",
#        "flausch_tokens",
#        "flausch_ratio",
#        "flausch_emoticons",
#        "flausch_emojis",
#        "extra_chars",
#        "all_caps"
        ]

log_reg_eval(eval_features, columns)

{'precision': 0.9444444444444444, 'recall': 0.9077669902912622, 'f1-score': 0.9257425742574257, 'support': 206.0}


In [60]:
columns = [
        "gbert-large_yes_comment",
        "bert-base-german-cased_yes_comment",
        "bert-base-german-cased_yes_spelling_corrected",
        "gbert-large_yes_spelling_corrected",
        "roberta-large_yes_translated",
#        "sentiment_orig",
#        "sentiment_spelling_corrected",
#        "sentiment_translated",
#        "sentiment_anger", 
#        "sentiment_disgust",
#        "sentiment_fear", 
#        "sentiment_joy",
#        "sentiment_neutral",
#        "sentiment_sadness",
#        "sentiment_surprise",
        "flausch_words",
        "flausch_tokens",
        "flausch_ratio",
        "flausch_emoticons",
        "flausch_emojis",
        "extra_chars",
        "all_caps"
        ]

log_reg_eval(eval_features, columns)

{'precision': 0.9411764705882353, 'recall': 0.9320388349514563, 'f1-score': 0.9365853658536586, 'support': 206.0}


In [61]:
columns = [
        "gbert-large_yes_comment",
        "bert-base-german-cased_yes_comment",
        "bert-base-german-cased_yes_spelling_corrected",
        "gbert-large_yes_spelling_corrected",
        "roberta-large_yes_translated",
        "sentiment_orig",
        "sentiment_spelling_corrected",
        "sentiment_translated",
        "sentiment_anger", 
        "sentiment_disgust",
        "sentiment_fear", 
        "sentiment_joy",
        "sentiment_neutral",
        "sentiment_sadness",
        "sentiment_surprise",
#        "flausch_words",
#        "flausch_tokens",
#        "flausch_ratio",
#        "flausch_emoticons",
#        "flausch_emojis",
#        "extra_chars",
#        "all_caps"
        ]

log_reg_eval(eval_features, columns)

{'precision': 0.9396984924623115, 'recall': 0.9077669902912622, 'f1-score': 0.9234567901234568, 'support': 206.0}


## Detailed feature combinations

In [62]:
# used in submission

columns = [
        "gbert-large_yes_comment",
#        "bert-base-german-cased_yes_comment",
#        "bert-base-german-cased_yes_spelling_corrected",
#        "gbert-large_yes_spelling_corrected",
#        "roberta-large_yes_translated",
        "sentiment_orig",
        "sentiment_spelling_corrected",
        "sentiment_translated",
        "sentiment_anger", 
        "sentiment_disgust",
        "sentiment_fear", 
        "sentiment_joy", 
        "sentiment_neutral",
        "sentiment_sadness",
        "sentiment_surprise",
        "flausch_words",
        "flausch_tokens",
        "flausch_ratio",
#        "flausch_emoticons",
#        "flausch_emojis",
#        "extra_chars",
#        "all_caps"
        ]
log_reg_eval(eval_features, columns)

{'precision': 0.9285714285714286, 'recall': 0.9466019417475728, 'f1-score': 0.9375, 'support': 206.0}


In [63]:
columns = [
        "gbert-large_yes_comment",
#        "bert-base-german-cased_yes_comment",
#        "bert-base-german-cased_yes_spelling_corrected",
#        "gbert-large_yes_spelling_corrected",
#        "roberta-large_yes_translated",
        "sentiment_orig",
#        "sentiment_spelling_corrected",
#        "sentiment_translated",
#        "sentiment_anger", 
#        "sentiment_disgust",
        "sentiment_fear", 
#        "sentiment_joy", 
#        "sentiment_neutral",
#        "sentiment_sadness",
#        "sentiment_surprise",
        "flausch_words",
#        "flausch_tokens",
#        "flausch_ratio",
#        "flausch_emoticons",
#        "flausch_emojis",
#        "extra_chars",
#        "all_caps"
        ]

log_reg_eval(eval_features, columns)

{'precision': 0.9285714285714286, 'recall': 0.9466019417475728, 'f1-score': 0.9375, 'support': 206.0}


## With the test data

Evaluation data for training log reg, test data for testing

In [64]:
def get_test_features(word_list, token_list, tokenizer, comment_version, path=path_test, data=test_task1):

    # features contain transformer predictions
    test_features = pd.read_csv(path + "features.csv")

    sentiment_orig = []
    sentiment_spelling_corrected = []
    sentiment_translated = []
    sentiment_anger = []
    sentiment_disgust = []
    sentiment_fear = []
    sentiment_joy = []
    sentiment_neutral = []
    sentiment_sadness = []
    sentiment_surprise = []
    flausch_words = []
    flausch_tokens = []
    flausch_ratio = []
    flausch_emoticons = []
    flausch_emojis = []
    extra_characters = []
    all_capitalized = []


    for i in range(len(test_features)):

        document = test_features.iloc[i]["document"]
        comment_id = test_features.iloc[i]["comment_id"]

        entry = data[(data["document"] == document) & (data["comment_id"] == comment_id)]

        orig_comment = entry["comment"].item()
        comment = entry[comment_version].item()
        tokens = tokenizer.tokenize(comment)


        # add sentiments
        sentiment_orig.append(entry["sentiment_orig"].item())
        sentiment_spelling_corrected.append(entry["sentiment_spelling_corrected"].item())
        sentiment_translated.append(entry["sentiment_translated"].item())
        sentiment_anger.append(entry["sentiment_anger"].item())
        sentiment_disgust.append(entry["sentiment_disgust"].item())
        sentiment_fear.append(entry["sentiment_fear"].item())
        sentiment_joy.append(entry["sentiment_joy"].item())
        sentiment_neutral.append(entry["sentiment_neutral"].item())
        sentiment_sadness.append(entry["sentiment_sadness"].item())
        sentiment_surprise.append(entry["sentiment_surprise"].item())


        # add counts
        flausch_word_count = 0
        for w in word_list:
            if w.lower() in comment.lower():
                flausch_word_count += 1
        flausch_words.append(flausch_word_count)

        flausch_token_count = 0
        for t in token_list:
            if t in tokens:
                flausch_token_count += 1
        flausch_tokens.append(flausch_token_count)

        ratio = 0
        if len(tokens) != 0:
            ratio = flausch_token_count / len(tokens)
        flausch_ratio.append(ratio)

        emoticon_count = 0
        for e in positive_emoticons:
            if e in orig_comment:
                emoticon_count += 1
        flausch_emoticons.append(emoticon_count)

        emoji_count = 0
        for e in positive_emojis:
            if e in orig_comment:
                emoji_count += 1
        flausch_emojis.append(emoji_count)

        all_capitalized.append(all_caps(orig_comment))
        extra_characters.append(extra_chars(orig_comment))

    
    test_features["sentiment_orig"] = sentiment_orig
    test_features["sentiment_spelling_corrected"] = sentiment_spelling_corrected
    test_features["sentiment_translated"] = sentiment_translated
    test_features["sentiment_anger"] = sentiment_anger
    test_features["sentiment_disgust"] = sentiment_disgust
    test_features["sentiment_fear"] = sentiment_fear
    test_features["sentiment_joy"] = sentiment_joy
    test_features["sentiment_neutral"] = sentiment_neutral
    test_features["sentiment_sadness"] = sentiment_sadness
    test_features["sentiment_surprise"] = sentiment_surprise
    test_features["flausch_words"] = flausch_words
    test_features["flausch_tokens"] = flausch_tokens
    test_features["flausch_ratio"] = flausch_ratio
    test_features["flausch_emoticons"] = flausch_emoticons
    test_features["flausch_emojis"] = flausch_emojis
    test_features["extra_chars"] = extra_characters
    test_features["all_caps"] = all_capitalized

    return test_features

In [65]:
def log_reg_test(eval_features, test_features, columns, eval_data=eval_task1, test_data=test_task1):

    train_X = []
    train_y = []

    # training on evaluation data
    for i in range(len(eval_features)):

        id = eval_features.iloc[i]["id"]
        flausch = eval_data[eval_data["id"] == id]["flausch"].item()
        train_y.append(get01(flausch))

        feature_vec = []

        for c in columns:
            feature_vec.append(eval_features.iloc[i][c])            

        train_X.append(feature_vec)
    
    test_X = []
    test_y = []

    # testing on test data
    for i in range(len(test_features)):

        document = test_features.iloc[i]["document"]
        comment_id = test_features.iloc[i]["comment_id"]
        flausch = test_data[(test_data["document"] == document) & (test_data["comment_id"] == comment_id)]["flausch"].item()
        test_y.append(get01(flausch))

        feature_vec = []

        for c in columns:
            feature_vec.append(test_features.iloc[i][c])            

        test_X.append(feature_vec)
    

    log_reg = LogisticRegression(max_iter=200).fit(train_X, train_y)
    pred_y = log_reg.predict(test_X)
    scores = classification_report(test_y, pred_y, output_dict=True, zero_division=0.0)
    print(scores["1"])

In [66]:
word_list = positive_words_orig_cut
comment_version = "comment"
tokenizer = tokenizer_ger
percentage_not = 1.25

token_list = create_token_list(tokenizer, comment_version, percentage_not, 3)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)
test_features = get_test_features(word_list, token_list, tokenizer, comment_version)

In [67]:
columns = [
        "gbert-large_yes_comment",
        "bert-base-german-cased_yes_comment",
        "bert-base-german-cased_yes_spelling_corrected",
        "gbert-large_yes_spelling_corrected",
        "roberta-large_yes_translated",
        "sentiment_orig",
        "sentiment_spelling_corrected",
        "sentiment_translated",
        "sentiment_anger", 
        "sentiment_disgust",
        "sentiment_fear", 
        "sentiment_joy",
        "sentiment_neutral",
        "sentiment_sadness",
        "sentiment_surprise",
        "flausch_words",
        "flausch_tokens",
        "flausch_ratio",
        "flausch_emoticons",
        "flausch_emojis",
        "extra_chars",
        "all_caps"
        ]

log_reg_test(eval_features, test_features, columns)

{'precision': 0.9228120516499283, 'recall': 0.8447596532702916, 'f1-score': 0.882062534284147, 'support': 3807.0}


In [68]:
columns = [
        "gbert-large_yes_comment",
#        "bert-base-german-cased_yes_comment",
#        "bert-base-german-cased_yes_spelling_corrected",
#        "gbert-large_yes_spelling_corrected",
#        "roberta-large_yes_translated",
        "sentiment_orig",
        "sentiment_spelling_corrected",
        "sentiment_translated",
        "sentiment_anger", 
        "sentiment_disgust",
        "sentiment_fear", 
        "sentiment_joy", 
        "sentiment_neutral",
        "sentiment_sadness",
        "sentiment_surprise",
        "flausch_words",
        "flausch_tokens",
        "flausch_ratio",
#        "flausch_emoticons",
#        "flausch_emojis",
#        "extra_chars",
#        "all_caps"
        ]

log_reg_test(eval_features, test_features, columns)

{'precision': 0.9006497022198159, 'recall': 0.8739164696611506, 'f1-score': 0.8870817224370084, 'support': 3807.0}


In [69]:
columns = [
        "gbert-large_yes_comment",
#        "bert-base-german-cased_yes_comment",
#        "bert-base-german-cased_yes_spelling_corrected",
#        "gbert-large_yes_spelling_corrected",
#        "roberta-large_yes_translated",
        "sentiment_orig",
#        "sentiment_spelling_corrected",
#        "sentiment_translated",
#        "sentiment_anger", 
#        "sentiment_disgust",
        "sentiment_fear", 
#        "sentiment_joy", 
#        "sentiment_neutral",
#        "sentiment_sadness",
#        "sentiment_surprise",
        "flausch_words",
#        "flausch_tokens",
#        "flausch_ratio",
#        "flausch_emoticons",
#        "flausch_emojis",
#        "extra_chars",
#        "all_caps"
        ]

log_reg_test(eval_features, test_features, columns)

{'precision': 0.8999729656663963, 'recall': 0.874441817704229, 'f1-score': 0.8870237143618438, 'support': 3807.0}


In [70]:
columns = [
        "gbert-large_yes_comment",
#        "bert-base-german-cased_yes_comment",
#        "bert-base-german-cased_yes_spelling_corrected",
#        "gbert-large_yes_spelling_corrected",
#        "roberta-large_yes_translated",
        "sentiment_orig",
#        "sentiment_spelling_corrected",
#        "sentiment_translated",
        "sentiment_anger", 
        "sentiment_disgust",
        "sentiment_fear", 
        "sentiment_joy", 
        "sentiment_neutral",
        "sentiment_sadness",
        "sentiment_surprise",
        "flausch_words",
        "flausch_tokens",
        "flausch_ratio",
#        "flausch_emoticons",
#        "flausch_emojis",
#        "extra_chars",
#        "all_caps"
        ]

log_reg_test(eval_features, test_features, columns)

{'precision': 0.9006497022198159, 'recall': 0.8739164696611506, 'f1-score': 0.8870817224370084, 'support': 3807.0}


# Different classifier methods

In [71]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

In [72]:
word_list = positive_words_orig_cut
comment_version = "comment"
tokenizer = tokenizer_ger
percentage_not = 1.25

token_list = create_token_list(tokenizer, comment_version, percentage_not, 3)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)

columns = [
        "gbert-large_yes_comment",
#        "bert-base-german-cased_yes_comment",
#        "bert-base-german-cased_yes_spelling_corrected",
#        "gbert-large_yes_spelling_corrected",
#        "roberta-large_yes_translated",
        "sentiment_orig",
        "sentiment_spelling_corrected",
        "sentiment_translated",
        "sentiment_anger", 
        "sentiment_disgust",
        "sentiment_fear", 
        "sentiment_joy",
        "sentiment_neutral",
        "sentiment_sadness",
        "sentiment_surprise",
        "flausch_words",
        "flausch_tokens",
        "flausch_ratio",
#        "flausch_emoticons",
#        "flausch_emojis",
#        "extra_chars",
#        "all_caps"
        ]

In [73]:
X = []
y = []

for i in range(len(eval_features)):

    id = eval_features.iloc[i]["id"]
    flausch = eval_task1[eval_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []

    for c in columns:
        feature_vec.append(eval_features.iloc[i][c])            

    X.append(feature_vec)

train_X, test_X, train_y, test_y =  train_test_split(X, y, test_size=0.2, random_state=42)

In [74]:
log_reg = LogisticRegression(max_iter=200).fit(train_X, train_y)
pred_y = log_reg.predict(test_X)
scores = classification_report(test_y, pred_y, output_dict=True, zero_division=0.0)
print(scores["1"])

{'precision': 0.9285714285714286, 'recall': 0.9466019417475728, 'f1-score': 0.9375, 'support': 206.0}


In [75]:
svm_reg = svm.SVR().fit(train_X, train_y)
pred_y = svm_reg.predict(test_X)
pred_y = list(map(lambda x: 1 if x > 0.5 else 0, list(pred_y)))
scores = classification_report(test_y, pred_y, output_dict=True, zero_division=0.0)
print(scores["1"])

{'precision': 0.9069767441860465, 'recall': 0.9466019417475728, 'f1-score': 0.9263657957244654, 'support': 206.0}


In [83]:
rf_clf = RandomForestClassifier(random_state=42).fit(train_X, train_y)
pred_y = rf_clf.predict(test_X)
scores = classification_report(test_y, pred_y, output_dict=True, zero_division=0.0)
print(scores["1"])

{'precision': 0.913068021450748, 'recall': 0.8497504596795377, 'f1-score': 0.8802721088435373, 'support': 3807.0}


In [84]:
mlp_clf = MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 2), random_state=42).fit(train_X, train_y)
pred_y = mlp_clf.predict(test_X)
scores = classification_report(test_y, pred_y, output_dict=True, zero_division=0.0)
print(scores["1"])

{'precision': 0.8998917748917749, 'recall': 0.8736537956396112, 'f1-score': 0.8865787018525924, 'support': 3807.0}


c:\Users\LaraE\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


# Splitting tranformer prediction & log reg with other features

Separate log reg with non-transformer features for comments predicted to be flausch / not flausch by transformer

In [78]:
word_list = positive_words_orig_cut
comment_version = "comment"
tokenizer = tokenizer_ger
percentage_not = 1.25

token_list = create_token_list(tokenizer, comment_version, percentage_not, 3)
eval_features = get_eval_features(word_list, token_list, tokenizer, comment_version)

In [79]:
# log reg split for BERT prdiction flausch (> 0.5) / not flausch

X0 = []
y0 = []
X1 = []
y1 = []

for i in range(len(eval_features)):

    id = eval_features.iloc[i]["id"]
    flausch = eval_task1[eval_task1["id"] == id]["flausch"].item()

    prediction = eval_features.iloc[i]["gbert-large_yes_comment"]

    feature_vec = []

    for c in [
        "sentiment_orig",
        "sentiment_spelling_corrected",
        "sentiment_translated",
        "sentiment_anger", 
        "sentiment_disgust",
        "sentiment_fear", 
        "sentiment_joy",
        "sentiment_neutral",
        "sentiment_sadness",
        "sentiment_surprise",
        "flausch_words",
        "flausch_tokens",
        "flausch_ratio",
        "flausch_emoticons",
        "flausch_emojis",
        "extra_chars",
        "all_caps"
        ]:
        feature_vec.append(eval_features.iloc[i][c])

    if prediction > 0.5:
        y1.append(get01(flausch))
        X1.append(feature_vec)
    else:
        y0.append(get01(flausch))
        X0.append(feature_vec)


train_X0, test_X0, train_y0, test_y0 =  train_test_split(X0, y0, test_size=0.2, random_state=42)
log_reg = LogisticRegression(max_iter=200).fit(train_X0, train_y0)
pred_y0 = log_reg.predict(test_X0)
print(classification_report(test_y0, pred_y0, zero_division=0.0))

train_X1, test_X1, train_y1, test_y1 =  train_test_split(X1, y1, test_size=0.2, random_state=42)
log_reg = LogisticRegression(max_iter=200).fit(train_X1, train_y1)
pred_y1 = log_reg.predict(test_X1)
print(classification_report(test_y1, pred_y1, zero_division=0.0))

test_y = test_y0 + test_y1
pred_y = list(pred_y0) + list(pred_y1)
print(classification_report(test_y, pred_y, zero_division=0.0))


              precision    recall  f1-score   support

           0       0.97      1.00      0.99       507
           1       0.00      0.00      0.00        14

    accuracy                           0.97       521
   macro avg       0.49      0.50      0.49       521
weighted avg       0.95      0.97      0.96       521

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        32
           1       0.86      1.00      0.92       189

    accuracy                           0.86       221
   macro avg       0.43      0.50      0.46       221
weighted avg       0.73      0.86      0.79       221

              precision    recall  f1-score   support

           0       0.97      0.94      0.96       539
           1       0.86      0.93      0.89       203

    accuracy                           0.94       742
   macro avg       0.91      0.94      0.92       742
weighted avg       0.94      0.94      0.94       742



# Saving the data for the best configuration

* word list cut from original GPT list
* token list created from data
* complete eval features
* complete test features
* predictions on test data

In [80]:
columns = [
        "gbert-large_yes_comment",
#        "bert-base-german-cased_yes_comment",
#        "bert-base-german-cased_yes_spelling_corrected",
#        "gbert-large_yes_spelling_corrected",
#        "roberta-large_yes_translated",
        "sentiment_orig",
        "sentiment_spelling_corrected",
        "sentiment_translated",
        "sentiment_anger", 
        "sentiment_disgust",
        "sentiment_fear", 
        "sentiment_joy",
        "sentiment_neutral",
        "sentiment_sadness",
        "sentiment_surprise",
        "flausch_words",
        "flausch_tokens",
        "flausch_ratio",
#        "flausch_emoticons",
#        "flausch_emojis",
#        "extra_chars",
#        "all_caps"
        ]

In [85]:
# create word/token lists & feature lists
positive_words_orig_cut = cut_lists(positive_words_orig, "comment", 1.5)
tokens_from_data = create_token_list(tokenizer_ger, "comment", 1.25, 3)
eval_features = get_eval_features(positive_words_orig_cut, tokens_from_data, tokenizer_ger, "comment")
test_features = get_test_features(positive_words_orig_cut, tokens_from_data, tokenizer_ger, "comment")

path_thesis = "Data/"

# save word & token lists as pkl files
with open(path_thesis + "positive_words.pkl", "wb") as outfile: 
   pickle.dump(positive_words_orig_cut, outfile)
with open(path_thesis + "positive_tokens_cut.pkl", "wb") as outfile: 
   pickle.dump(tokens_from_data, outfile)

# save feature lists as csv files
eval_features.to_csv(path_thesis + "eval_features.csv", index=False)
test_features.to_csv(path_thesis + "test_features.csv", index=False)

In [82]:
# get and save predictions for the best configuration
# for using in combination with subtask 2

train_X = []
train_y = []

for i in range(len(eval_features)):

    id = eval_features.iloc[i]["id"]
    flausch = eval_task1[eval_task1["id"] == id]["flausch"].item()
    train_y.append(get01(flausch))

    feature_vec = []

    for c in columns:
        feature_vec.append(eval_features.iloc[i][c])            

    train_X.append(feature_vec)


test_X = []
test_y = []

for i in range(len(test_features)):

    document = test_features.iloc[i]["document"]
    comment_id = test_features.iloc[i]["comment_id"]
    flausch = test_task1[(test_task1["document"] == document) & (test_task1["comment_id"] == comment_id)]["flausch"].item()
    test_y.append(get01(flausch))

    feature_vec = []

    for c in columns:
        feature_vec.append(test_features.iloc[i][c])            

    test_X.append(feature_vec)


log_reg = LogisticRegression(max_iter=200).fit(train_X, train_y)
pred_y = log_reg.predict(test_X)

predictions = list(map(lambda x: "yes" if x == 1 else "no", list(pred_y)))

with open(path_thesis + "subtask1_test_predictions.pkl", "wb") as outfile: 
   pickle.dump(predictions, outfile)